In [ ]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.split import make_time_folds, expanding_window_splits
from src.features.build_features import build_features

DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PREDICTIONS = PROJECT_ROOT / "data" / "predictions"
REPORTS_FIGURES = PROJECT_ROOT / "reports" / "figures"

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.figsize"] = (10, 5)

print("Imports OK")

In [ ]:
# 1. Загружаем train и test
train = pd.read_parquet(DATA_INTERIM / "train.parquet", engine="pyarrow")
test = pd.read_parquet(DATA_INTERIM / "test.parquet", engine="pyarrow")

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")
print()

# 2. Патч колонок в test: id-XX -> id_XX (из урока дня 4)
rename_map = {c: c.replace("id-", "id_") for c in test.columns if c.startswith("id-")}
test = test.rename(columns=rename_map)
print(f"Patched test columns: {len(rename_map)}")
print()

# 3. Feature engineering — три блока из src/features/
t0 = time.time()
train, test = build_features(train, test)
print(f"\nFE total time: {time.time() - t0:.1f}s")
print()
print(f"Train after FE: {train.shape}")
print(f"Test after FE:  {test.shape}")

In [ ]:
new_cols = [
    "day", "hour", "dayofweek", "is_night",
    "log_amt", "amt_cents", "amt_has_cents",
    "card1_count", "card1_amt_mean", "card1_amt_std", "card1_amt_max",
    "card1_nunique_productcd", "amt_to_card1_mean_ratio",
]

print("Типы новых фичей:")
print(train[new_cols].dtypes)
print()

print("Статистики:")
print(train[new_cols].describe().T[["count", "mean", "std", "min", "max"]])
print()

# Проверяем, что пропусков в ключевых фичах нет
print("Пропуски в новых фичах:")
print(train[new_cols].isna().sum()[train[new_cols].isna().sum() > 0])

In [ ]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

# Фолды — те же, что в baseline v0
folds = make_time_folds(train["TransactionDT"], n_splits=5)

# Подготовка X / y
drop_cols = ["TransactionID", "isFraud", "TransactionDT"]
y = train["isFraud"].astype(np.int8)
X = train.drop(columns=drop_cols)

cat_cols = X.select_dtypes(include=["category"]).columns.tolist()
print(f"X: {X.shape}, cat cols: {len(cat_cols)}")

# Те же гиперпараметры, что в baseline v0 — чтобы изолировать эффект FE
lgb_params = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_child_samples": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "n_jobs": -1,
    "random_state": 42,
}

splits = expanding_window_splits(folds, n_splits=5)
oof_preds = np.zeros(len(X), dtype=np.float32)
cv_scores = []
models = []

for fold_num, (train_idx, valid_idx) in enumerate(splits, start=1):
    t0 = time.time()
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[valid_idx], y.iloc[valid_idx]

    train_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols)
    valid_data = lgb.Dataset(X_va, label=y_va, categorical_feature=cat_cols,
                              reference=train_data)

    model = lgb.train(
        params=lgb_params,
        train_set=train_data,
        num_boost_round=2000,
        valid_sets=[valid_data],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )

    oof_preds[valid_idx] = model.predict(X_va, num_iteration=model.best_iteration)
    fold_auc = roc_auc_score(y_va, oof_preds[valid_idx])
    cv_scores.append(fold_auc)
    models.append(model)

    elapsed = time.time() - t0
    print(f"Fold {fold_num}: AUC = {fold_auc:.5f} "
          f"(best_iter={model.best_iteration}, time={elapsed:.0f}s)")

print()
print(f"CV AUC: {np.mean(cv_scores):.5f} ± {np.std(cv_scores):.5f}")
valid_mask = folds > 0
print(f"OOF AUC (only validated): {roc_auc_score(y[valid_mask], oof_preds[valid_mask]):.5f}")

# Сравнение с baseline v0
baseline_v0 = 0.90742
delta = np.mean(cv_scores) - baseline_v0
print()
print(f"Baseline v0 CV:     {baseline_v0:.5f}")
print(f"v1 (with FE) CV:    {np.mean(cv_scores):.5f}")
print(f"Delta:              {delta:+.5f}")

In [ ]:
# Выравниваем колонки test к train
X_test = test.drop(columns=["TransactionID", "TransactionDT"])
X_test = X_test[X.columns]

# Согласуем dtypes категорий
for col in cat_cols:
    if col in X_test.columns:
        X_test[col] = X_test[col].astype(X[col].dtype)

test_preds = np.zeros(len(X_test), dtype=np.float32)
for i, model in enumerate(models):
    p = model.predict(X_test, num_iteration=model.best_iteration)
    test_preds += p / len(models)

print(f"Predictions: min={test_preds.min():.4f}, max={test_preds.max():.4f}, "
      f"mean={test_preds.mean():.4f}")

# Сохраняем submission + OOF
submission = pd.DataFrame({
    "TransactionID": test["TransactionID"],
    "isFraud": test_preds,
})
submission.to_csv(DATA_PREDICTIONS / "sub_lgbm_fe_v1.csv", index=False)

oof_df = pd.DataFrame({
    "TransactionID": train["TransactionID"],
    "isFraud_true": y,
    "isFraud_pred": oof_preds,
    "fold": folds,
})
oof_df.to_csv(DATA_PREDICTIONS / "oof_lgbm_fe_v1.csv", index=False)

print("Saved: sub_lgbm_fe_v1.csv + oof_lgbm_fe_v1.csv")